In [5]:
# train, test 7:3으로 나누기
# 들어오는 인자는 data -> dataframe으로 들어옴
# 라벨에 해당하는 label들
import pandas as pd

def stratified_split(df, label_col):

    # 라벨별로 데이터를 분리한 뒤, 리스트에 저장할 공간
    train_list = []
    test_list = []

    # 라벨 컬럼의 고유값(예: 0, 1, 2 등) 하나씩 반복
    for label in df[label_col].unique():
        # 데이터 프레임 하나씩 진행
        # 해당 라벨에 해당하는 데이터만 추출
        label_df = df[df[label_col] == label] #라벨별로 데이터프레임 생성
        # 테스트 데이터 개수 계산 (비율대로)
        n_test = max(1, int(len(label_df) * 0.3))  # 최소 1개는 테스트에 포함

        # 앞부분을 테스트, 나머지는 학습으로 나누기
        test_list.append(label_df[:n_test])
        train_list.append(label_df[n_test:])

    # 리스트에 쌓아둔 라벨별 데이터들을 모두 합치기
    train_df = pd.concat(train_list).reset_index(drop=True)
    test_df = pd.concat(test_list).reset_index(drop=True)

    return train_df, test_df

In [6]:
# 예시 데이터프레임 만들기
df = pd.DataFrame({
    'text': ['a', 'b', 'c', 'd', 'e', 'f'],
    'label': [0, 0, 0, 1, 1, 1]
})

# 라벨 비율에 맞춰 분할
train, test = stratified_split(df, label_col='label')

print("Train:\n", train)
print("\nTest:\n", test)

Train:
   text  label
0    b      0
1    c      0
2    e      1
3    f      1

Test:
   text  label
0    a      0
1    d      1


# knn: k-최근접 이웃 모델, 지도학습, 분류에 사용

In [ ]:
# 모델 불러오기
from sklearn.datasets import load_iris
iris = load_iris()
print(iris.DESCR)

.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ====================
                Min  Max   Mean    SD   Class Correlation
============== ==== ==== ======= ===== ====================
sepal length:   4.3  7.9   5.84   0.83    0.7826
sepal width:    2.0  4.4   3.05   0.43   -0.4194
petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
============== ==== ==== ======= ===== ====================

:Missing Attribute Values: None
:Class Distribution: 33.3% for each of 3 classes.
:Cr

In [8]:
# 데이터 확인
data = iris.data
target = iris.target
print(data)
print(target)

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]
 [5.4 3.9 1.7 0.4]
 [4.6 3.4 1.4 0.3]
 [5.  3.4 1.5 0.2]
 [4.4 2.9 1.4 0.2]
 [4.9 3.1 1.5 0.1]
 [5.4 3.7 1.5 0.2]
 [4.8 3.4 1.6 0.2]
 [4.8 3.  1.4 0.1]
 [4.3 3.  1.1 0.1]
 [5.8 4.  1.2 0.2]
 [5.7 4.4 1.5 0.4]
 [5.4 3.9 1.3 0.4]
 [5.1 3.5 1.4 0.3]
 [5.7 3.8 1.7 0.3]
 [5.1 3.8 1.5 0.3]
 [5.4 3.4 1.7 0.2]
 [5.1 3.7 1.5 0.4]
 [4.6 3.6 1.  0.2]
 [5.1 3.3 1.7 0.5]
 [4.8 3.4 1.9 0.2]
 [5.  3.  1.6 0.2]
 [5.  3.4 1.6 0.4]
 [5.2 3.5 1.5 0.2]
 [5.2 3.4 1.4 0.2]
 [4.7 3.2 1.6 0.2]
 [4.8 3.1 1.6 0.2]
 [5.4 3.4 1.5 0.4]
 [5.2 4.1 1.5 0.1]
 [5.5 4.2 1.4 0.2]
 [4.9 3.1 1.5 0.2]
 [5.  3.2 1.2 0.2]
 [5.5 3.5 1.3 0.2]
 [4.9 3.6 1.4 0.1]
 [4.4 3.  1.3 0.2]
 [5.1 3.4 1.5 0.2]
 [5.  3.5 1.3 0.3]
 [4.5 2.3 1.3 0.3]
 [4.4 3.2 1.3 0.2]
 [5.  3.5 1.6 0.6]
 [5.1 3.8 1.9 0.4]
 [4.8 3.  1.4 0.3]
 [5.1 3.8 1.6 0.2]
 [4.6 3.2 1.4 0.2]
 [5.3 3.7 1.5 0.2]
 [5.  3.3 1.4 0.2]
 [7.  3.2 4.7 1.4]
 [6.4 3.2 4.5 1.5]
 [6.9 3.1 4.

In [9]:
# 데이터 크기 출력
print(data.shape, target.shape)

(150, 4) (150,)


In [10]:
import numpy as np
# 고유 라벨값과 라벨당 데이터 개수 출력
np.unique(target, return_counts=True)

(array([0, 1, 2]), array([50, 50, 50]))

In [11]:
# 7:3 비율로 데이터 스플릿
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(data,
                                                    target,
                                                    test_size=0.3,
                                                    random_state=0)

print(x_train.shape, y_test.shape)

(105, 4) (45,)


In [12]:
# 고유값 출력
np.unique(y_test, return_counts=True)

(array([0, 1, 2]), array([16, 18, 11]))

In [13]:
print(np.min(x_train, axis=0))
print(np.max(x_train, axis=0))
print(np.mean(x_train, axis=0))
print(np.std(x_train, axis=0))

[4.3 2.  1.1 0.1]
[7.9 4.4 6.9 2.5]
[5.89333333 3.0447619  3.82857143 1.22761905]
[0.87268242 0.43925932 1.79595918 0.77637684]


In [14]:
(x_train - np.mean(x_train, axis=0)) / np.std(x_train, axis=0)

array([[-1.02366372, -2.37846268, -0.18295039, -0.29318114],
       [ 0.69517462, -0.10190314,  0.93066067,  0.7372463 ],
       [ 0.92435306,  0.58106472,  1.04202177,  1.63887031],
       [ 0.1222285 , -1.92315077,  0.6522579 ,  0.35083601],
       [ 0.92435306, -1.24018291,  1.09770233,  0.7372463 ],
       [-0.33612839, -1.24018291,  0.03977182, -0.16437771],
       [ 2.07024529, -0.10190314,  1.26474398,  1.38126345],
       [ 0.46599617,  0.58106472,  0.48521625,  0.47963944],
       [-0.45071761, -1.46783886, -0.01590873, -0.16437771],
       [ 0.46599617, -0.784871  ,  0.59657735,  0.7372463 ],
       [ 0.46599617, -0.55721505,  0.70793846,  0.35083601],
       [-1.13825295, -1.24018291,  0.37385514,  0.60844287],
       [ 0.46599617, -1.24018291,  0.6522579 ,  0.86604973],
       [ 1.26812073,  0.35340877,  0.48521625,  0.22203258],
       [ 0.69517462, -0.10190314,  0.76361901,  0.99485316],
       [ 0.1222285 ,  0.80872067,  0.37385514,  0.47963944],
       [-1.25284217,  0.

In [15]:
# STANDARD SCALER 적용
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(x_train)
x_train_zst = scaler.transform(x_train)
x_test_zst = scaler.transform(x_test)
print(x_train[:5])
print(x_train_zst[:5])

[[5.  2.  3.5 1. ]
 [6.5 3.  5.5 1.8]
 [6.7 3.3 5.7 2.5]
 [6.  2.2 5.  1.5]
 [6.7 2.5 5.8 1.8]]
[[-1.02366372 -2.37846268 -0.18295039 -0.29318114]
 [ 0.69517462 -0.10190314  0.93066067  0.7372463 ]
 [ 0.92435306  0.58106472  1.04202177  1.63887031]
 [ 0.1222285  -1.92315077  0.6522579   0.35083601]
 [ 0.92435306 -1.24018291  1.09770233  0.7372463 ]]


In [16]:
# KNN 알고리즘 적용
from sklearn.neighbors import KNeighborsClassifier
iris_knn = KNeighborsClassifier(n_neighbors=10)
iris_knn.fit(x_train_zst, y_train)
pred = iris_knn.predict(x_test_zst)

In [17]:
# 실제 라벨값과 예측값 출력
print(y_test)
print(pred)

[2 1 0 2 0 2 0 1 1 1 2 1 1 1 1 0 1 1 0 0 2 1 0 0 2 0 0 1 1 0 2 1 0 2 2 1 0
 1 1 1 2 0 2 0 0]
[2 1 0 2 0 2 0 1 1 1 2 1 1 1 1 0 1 1 0 0 2 1 0 0 2 0 0 1 1 0 2 1 0 2 2 1 0
 2 1 1 2 0 2 0 0]


In [18]:
#평균 정확도 출력
print(np.mean(y_test == pred))

0.9777777777777777


In [ ]:

# K 개수 변하면서 예측값 확률 변화 확인인
for k in range(1,21):
    iris_knn = KNeighborsClassifier(n_neighbors=k)
    iris_knn.fit(x_train_zst, y_train)
    pred = iris_knn.predict(x_test_zst)
    print(k, "==>", np.mean(y_test == pred))

1 ==> 0.9333333333333333
2 ==> 0.9555555555555556
3 ==> 0.9777777777777777
4 ==> 0.9777777777777777
5 ==> 0.9777777777777777
6 ==> 0.9777777777777777
7 ==> 0.9777777777777777
8 ==> 0.9777777777777777
9 ==> 0.9777777777777777
10 ==> 0.9777777777777777
11 ==> 0.9777777777777777
12 ==> 0.9777777777777777
13 ==> 0.9777777777777777
14 ==> 0.9777777777777777
15 ==> 0.9777777777777777
16 ==> 0.9777777777777777
17 ==> 0.9555555555555556
18 ==> 0.9777777777777777
19 ==> 0.9333333333333333
20 ==> 0.9555555555555556
21 ==> 0.9333333333333333
22 ==> 0.9555555555555556
23 ==> 0.9333333333333333
24 ==> 0.9111111111111111
25 ==> 0.9111111111111111
26 ==> 0.9333333333333333
27 ==> 0.9333333333333333
28 ==> 0.9333333333333333
29 ==> 0.9333333333333333
30 ==> 0.9111111111111111
31 ==> 0.9333333333333333
32 ==> 0.9333333333333333
33 ==> 0.9333333333333333
34 ==> 0.9333333333333333
35 ==> 0.9111111111111111
36 ==> 0.8888888888888888
37 ==> 0.8888888888888888
38 ==> 0.8666666666666667
39 ==> 0.888888888888

In [29]:
# 마찬가지로 진행
iris_knn = KNeighborsClassifier(n_neighbors=13)
iris_knn.fit(x_train_zst, y_train)
pred = iris_knn.predict(x_test_zst)
print(k, "==>", np.mean(y_test == pred))

100 ==> 0.9777777777777777


In [30]:
# 컨퓨전 매트릭스 진행
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, pred)

array([[16,  0,  0],
       [ 0, 17,  1],
       [ 0,  0, 11]])

: 